In [1]:
import os
import re
import cv2
import urllib.parse
import numpy as np
import pandas as pd
from sqlalchemy import create_engine
from dotenv import load_dotenv

load_dotenv()
DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT")
DB_NAME = os.getenv("DB_NAME")

def extract_validation_dataset(df):
    # 로드된 데이터 프레임에서 조건별 랜덤 추출을 수행하는 함수
    error_types = ['NONE', '녹', '스크레치', '균열', '찍힘']
    sampled_list = []

    for etype in error_types:
        df_target = df[df['ERRORTYPE'] == etype]

        if len(df_target) < 1100:
            print(f"{etype} 데이터가 1100개 미만입니다. 현재 수량: {len(df_target)}")
            sampled = df_target
        else:
            sampled = df_target.sample(n=1100, random_state=42)

        sampled_list.append(sampled)

    df_errors = pd.concat(sampled_list)

    # 중복 방지를 위해 이미 추출된 데이터 제외
    df_remaining = df[~df.index.isin(df_errors.index)]

    # ISERROR가 null인 데이터 5500개 추출
    df_null_pool = df_remaining[df_remaining['ISERROR'].isnull()]

    if len(df_null_pool) < 5500:
        print(f"ISERROR가 NULL인 잔여 데이터가 5500개 미만입니다. 현재 수량: {len(df_null_pool)}")
        sampled_null = df_null_pool
    else:
        sampled_null = df_null_pool.sample(n=5500, random_state=42)

    # 최종 데이터셋 병합
    df_final = pd.concat([df_errors, sampled_null]).reset_index(drop=True)
    return df_final

def run_inference_pipeline():
    # 환경 변수 로드 및 DB 엔진 생성
    safe_password = urllib.parse.quote_plus(DB_PASSWORD)
    db_url = f"mysql+pymysql://{DB_USER}:{safe_password}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
    engine = create_engine(db_url)
    
    table_name = "ai_vision_davalue" 
    query = f"SELECT * FROM {table_name}"
    
    print("-> DB에서 원본 데이터를 조회하는 중...")
    try:
        # DB에서 전체 데이터 로드
        df_all = pd.read_sql(query, con=engine)
        print(f"-> 데이터 로드 완료 (총 행 수: {len(df_all)}개)")
    except Exception as e:
        print(f"[오류] DB 데이터를 불러오는 중 에러가 발생했습니다: {e}")
        return None
    
    print("-> 정합성 테스트용 11,000개 데이터 랜덤 추출 시작...")
    df_test = extract_validation_dataset(df_all)
    print(f"-> 최종 검증 데이터셋 준비 완료 (총 행 수: {len(df_test)}개)")
    
    return df_test

df_validation = run_inference_pipeline()

-> DB에서 원본 데이터를 조회하는 중...
-> 데이터 로드 완료 (총 행 수: 60540개)
-> 정합성 테스트용 11,000개 데이터 랜덤 추출 시작...
-> 최종 검증 데이터셋 준비 완료 (총 행 수: 11000개)


In [2]:
safe_password = urllib.parse.quote_plus(DB_PASSWORD)
db_url = f"mysql+pymysql://{DB_USER}:{safe_password}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(db_url)

In [46]:
def run_step1_structure_csv(df_validation, engine):
    print("-> [Step 1] 데이터 구조 정합성 검증 및 CSV 추출을 시작합니다.")
    
    img_results = []
    txt_results = []
    proc_results = []
    
    total_rows = len(df_validation)
    if total_rows == 0:
        print("[오류] 검증할 데이터프레임이 비어 있습니다.")
        return

    # 공정 DB 매칭을 위한 고유 LOTID 사전 로드
    validation_lots = df_validation['LOTID'].dropna().unique().tolist()
    total_val_lots = len(validation_lots)
    proc_distinct_lots = set()
    
    if total_val_lots > 0:
        format_strings = ','.join(['%s'] * total_val_lots)
        try:
            query = f"SELECT DISTINCT lotid FROM ai_proc_davalue WHERE lotid IN ({format_strings})"
            df_proc = pd.read_sql(query, con=engine, params=tuple(validation_lots))
            proc_distinct_lots = set(df_proc['lotid'].dropna().unique())
        except Exception as e:
            print(f"[오류] DB 조회 실패: {e}")

    # 통계 집계용 변수
    valid_filename_count = 0
    physical_exist_count = 0
    valid_text_count = 0
    total_text_target = 0
    valid_extensions = ['.jpg', '.jpeg', '.png']

    # 행별 전수 검사 루프
    for idx, row in df_validation.iterrows():
        lotid = row.get('LOTID')
        
        # -------------------------------------------------------------
        # 1. [이미지] 검증
        # -------------------------------------------------------------
        path = str(row.get('FILEPATH'))
        actual_stem, ext = os.path.splitext(os.path.basename(path))
        
        is_exist = os.path.exists(path)
        if is_exist:
            physical_exist_count += 1
            
        raw_datime = row.get('DATIME')
        is_format_match = False
        expected_stem = ""
        
        if pd.notnull(raw_datime) and pd.notnull(lotid):
            try:
                dt_str = pd.to_datetime(raw_datime).strftime('%Y%m%d%H%M%S')
                expected_stem = f"{dt_str}_{lotid}"
                if (actual_stem == expected_stem) and (ext.lower() in valid_extensions):
                    is_format_match = True
                    valid_filename_count += 1
            except:
                pass
                
        img_pass = "PASS" if (is_exist and is_format_match) else "FAIL"
        img_results.append({
            "LOTID": lotid,
            "FILEPATH": path,
            "EXPECTED_NAME": expected_stem,
            "ACTUAL_NAME": actual_stem,
            "FILE_EXISTS": is_exist,
            "FORMAT_MATCH": is_format_match,
            "RESULT": img_pass
        })

        # -------------------------------------------------------------
        # 2. [텍스트] 검증
        # -------------------------------------------------------------
        is_err_null = pd.isnull(row.get('ISERROR'))
        raw_err_text = row.get('ERRORTEXT')
        
        # 진짜 null값이거나, 공백만 있거나, 문자열 'nan'/'none' 인지 판별
        is_text_empty = pd.isnull(raw_err_text) or str(raw_err_text).strip() == "" or str(raw_err_text).strip().lower() in ['nan', 'none']
        
        if not is_err_null:
            # ISERROR가 존재하므로 검증 대상에 포함 (5500건 타겟)
            total_text_target += 1
            
            if not is_text_empty:
                valid_text_count += 1
                txt_pass = "PASS"
            else:
                txt_pass = "FAIL"
                
            txt_results.append({
                "LOTID": lotid,
                "ISERROR_NULL": is_err_null,
                "ERRORTEXT": raw_err_text,
                "RESULT": txt_pass
            })
        else:
            # ISERROR가 없으면 테스트 제외
            txt_results.append({
                "LOTID": lotid,
                "ISERROR_NULL": is_err_null,
                "ERRORTEXT": raw_err_text,
                "RESULT": "EXCLUDED"
            })

        # -------------------------------------------------------------
        # 3. [공정] 검증
        # -------------------------------------------------------------
        is_in_db = lotid in proc_distinct_lots
        proc_pass = "PASS" if is_in_db else "FAIL"
        
        proc_results.append({
            "LOTID": lotid,
            "EXIST_IN_PROC_DB": is_in_db,
            "RESULT": proc_pass
        })

    # 고유 LOTID 매칭 수 (비율 계산용)
    matched_lot_count = len(set(validation_lots).intersection(proc_distinct_lots))

    # 데이터프레임 변환 및 CSV 저장
    df_img = pd.DataFrame(img_results)
    df_txt = pd.DataFrame(txt_results)
    df_proc = pd.DataFrame(proc_results)
    
    # 작업 디렉토리에 CSV 저장 (한글 깨짐 방지 utf-8-sig)
    df_img.to_csv("./KOTCA_result/1_structure_image.csv", index=False, encoding="utf-8-sig")
    df_txt.to_csv("./KOTCA_result/1_structure_text.csv", index=False, encoding="utf-8-sig")
    df_proc.to_csv("./KOTCA_result/1_structure_process.csv", index=False, encoding="utf-8-sig")
    
    # 비율 및 점수 계산
    img_score = (((valid_filename_count / total_rows) * 100) + ((physical_exist_count / total_rows) * 100)) / 2
    txt_score = (valid_text_count / total_text_target) * 100 if total_text_target > 0 else 0.0
    proc_score = (matched_lot_count / total_val_lots * 100) if total_val_lots > 0 else 0.0
    total_score = (img_score + txt_score + proc_score) / 3

    print("-> [완료] 1_structure_image.csv, 1_structure_text.csv, 1_structure_process.csv 생성 완료\n")
    
    # 요약 및 기준 출력
    print("="*65)
    print(" [Step 1] 데이터 구조 정합성 검증 결과 리포트")
    print("="*65)
    
    print(f" [이미지] 정합성 점수: {img_score:>6.2f}%")
    print(f"   - 현황: PASS {len(df_img[df_img['RESULT']=='PASS'])}건 / FAIL {len(df_img[df_img['RESULT']=='FAIL'])}건")
    print("   - 기준: DATIME_LOTID 형식의 파일명 일치 여부(DATIME_LOTID) 및 물리 파일 존재 여부")
    print(f"   - 상세: 파일명 일치({valid_filename_count}/{total_rows}), 물리파일 존재({physical_exist_count}/{total_rows})\n")

    print(f" [텍스트] 정합성 점수: {txt_score:>6.2f}%")
    print(f"   - PASS {valid_text_count}건 / FAIL {total_text_target - valid_text_count}건 (검증 제외: {total_rows - total_text_target}건)")
    print("   - 기준: ISERROR(에러유무)가 존재할 때만 ERRORTEXT(에러내용)가 존재하는 조건부 매칭")
    print(f"   - 상세: 텍스트-에러 매칭({valid_text_count}/{total_text_target})\n")

    print(f" [공정]   정합성 점수: {proc_score:>6.2f}%")
    print(f"   - 현황: PASS {len(df_proc[df_proc['RESULT']=='PASS'])}건 / FAIL {len(df_proc[df_proc['RESULT']=='FAIL'])}건")
    print("   - 기준: 비전 검증 데이터의 LOTID가 공정 DB(ai_proc_davalue)에 1:1 매핑되는지 확인")
    print(f"   - 상세: 고유 LOTID 매칭({matched_lot_count}/{total_val_lots})\n")
    
    print("-" * 65)
    print(f" >>> [데이터 구조] 전체 합산 정합성 평균 : {total_score:>6.2f}%")
    print("=" * 65 + "\n")

    return total_score

# 실행부
score_1 = run_step1_structure_csv(df_validation, engine)

-> [Step 1] 데이터 구조 정합성 검증 및 CSV 추출을 시작합니다.
-> [완료] 1_structure_image.csv, 1_structure_text.csv, 1_structure_process.csv 생성 완료

 [Step 1] 데이터 구조 정합성 검증 결과 리포트
 [이미지] 정합성 점수: 100.00%
   - 현황: PASS 11000건 / FAIL 0건
   - 기준: DATIME_LOTID 형식의 파일명 일치 여부(DATIME_LOTID) 및 물리 파일 존재 여부
   - 상세: 파일명 일치(11000/11000), 물리파일 존재(11000/11000)

 [텍스트] 정합성 점수: 100.00%
   - PASS 5500건 / FAIL 0건 (검증 제외: 5500건)
   - 기준: ISERROR(에러유무)가 존재할 때만 ERRORTEXT(에러내용)가 존재하는 조건부 매칭
   - 상세: 텍스트-에러 매칭(5500/5500)

 [공정]   정합성 점수: 100.00%
   - 현황: PASS 11000건 / FAIL 0건
   - 기준: 비전 검증 데이터의 LOTID가 공정 DB(ai_proc_davalue)에 1:1 매핑되는지 확인
   - 상세: 고유 LOTID 매칭(11000/11000)

-----------------------------------------------------------------
 >>> [데이터 구조] 전체 합산 정합성 평균 : 100.00%



In [47]:
def run_step2_input_range_csv(df_validation, engine):
    print("-> [Step 2] 입력값 범위 정합성 검증 및 CSV 추출을 시작합니다.")
    
    img_results = []
    txt_results = []
    proc_results = []
    
    total_rows = len(df_validation)
    if total_rows == 0:
        print("[오류] 검증할 데이터프레임이 비어 있습니다.")
        return

    # -------------------------------------------------------------
    # [사전 준비] 공정 DB 결측치 및 이상치 일괄 판별 (속도 최적화)
    # -------------------------------------------------------------
    validation_lots = df_validation['LOTID'].dropna().unique().tolist()
    total_val_lots = len(validation_lots)
    
    existing_lots = set()
    invalid_lots = set()
    total_proc_rows = 0
    valid_proc_count = 0
    
    if total_val_lots > 0:
        format_strings = ','.join(['%s'] * total_val_lots)
        try:
            query = f"SELECT lotid, PROC_TYPE, WELD_CURR, WELD_VOLT, WELD_TEMP, PAINT_PRESS, PAINT_TEMP, PAINT_HUMID FROM ai_proc_davalue WHERE lotid IN ({format_strings})"
            df_proc = pd.read_sql(query, con=engine, params=tuple(validation_lots))
            total_proc_rows = len(df_proc)
            
            if total_proc_rows > 0:
                existing_lots = set(df_proc['lotid'].dropna().unique())
                
                # 결측치 조건
                cond_weld_missing = (df_proc['PROC_TYPE'] == '용접') & df_proc[['WELD_CURR', 'WELD_VOLT', 'WELD_TEMP']].isnull().any(axis=1)
                cond_paint_missing = (df_proc['PROC_TYPE'] == '도장') & df_proc[['PAINT_PRESS', 'PAINT_TEMP', 'PAINT_HUMID']].isnull().any(axis=1)
                
                # 이상치 조건 (요청하신 범위 적용)
                cond_weld_outlier = (df_proc['PROC_TYPE'] == '용접') & (((df_proc['WELD_CURR'] < 250) | (df_proc['WELD_CURR'] > 350)) | ((df_proc['WELD_VOLT'] < 24) | (df_proc['WELD_VOLT'] > 34)) | ((df_proc['WELD_TEMP'] < 100) | (df_proc['WELD_TEMP'] > 150)))
                cond_paint_outlier = (df_proc['PROC_TYPE'] == '도장') & (((df_proc['PAINT_PRESS'] < 2) | (df_proc['PAINT_PRESS'] > 6)) | ((df_proc['PAINT_TEMP'] < 20) | (df_proc['PAINT_TEMP'] > 30)) | ((df_proc['PAINT_HUMID'] < 40) | (df_proc['PAINT_HUMID'] > 70)))
                
                # 전체 불량 행 판별
                is_invalid = cond_weld_missing | cond_paint_missing | cond_weld_outlier | cond_paint_outlier
                
                # 120만 건 중 정상 통과 건수 산출
                valid_proc_count = (~is_invalid).sum()
                
                # 불량이 1건이라도 존재하는 LOTID 추출
                df_proc['IS_INVALID'] = is_invalid
                invalid_lots = set(df_proc[df_proc['IS_INVALID']]['lotid'].dropna().unique())
                
        except Exception as e:
            print(f"[오류] DB 조회 실패: {e}")

    # 통계 집계용 변수
    valid_pixel_count = 0
    valid_keyword_count = 0
    total_text_target = 0
    keyword_pattern = re.compile(r'녹|찍힘|균열|스크레치')

    # 행별 전수 검사 루프
    for idx, row in df_validation.iterrows():
        lotid = row.get('LOTID')
        
        # -------------------------------------------------------------
        # 1. [이미지] 픽셀 값 확인
        # -------------------------------------------------------------
        path = str(row.get('FILEPATH'))
        is_valid_pixel = False
        
        if os.path.exists(path):
            img = cv2.imread(path)
            if img is not None:
                is_valid_pixel = True
                valid_pixel_count += 1
                
        img_pass = "PASS" if is_valid_pixel else "FAIL"
        img_results.append({
            "LOTID": lotid,
            "FILEPATH": path,
            "PIXEL_VALID": is_valid_pixel,
            "RESULT": img_pass
        })

        # -------------------------------------------------------------
        # 2. [텍스트] 사전 정의 키워드 확인
        # -------------------------------------------------------------
        etype = str(row.get('ERRORTYPE')).strip().upper()
        etext = str(row.get('ERRORTEXT')).strip()
        
        is_real_error = (etype != 'NONE' and etype != 'NAN' and pd.notnull(row.get('ERRORTYPE')))
        
        if is_real_error:
            total_text_target += 1
            has_keyword = False
            if pd.notnull(etext) and etext != "":
                if keyword_pattern.search(etext):
                    has_keyword = True
                    valid_keyword_count += 1
            
            txt_pass = "PASS" if has_keyword else "FAIL"
            txt_results.append({
                "LOTID": lotid,
                "ERRORTYPE": etype,
                "ERRORTEXT": etext,
                "HAS_KEYWORD": has_keyword,
                "RESULT": txt_pass
            })
        else:
            txt_results.append({
                "LOTID": lotid,
                "ERRORTYPE": etype,
                "ERRORTEXT": etext,
                "HAS_KEYWORD": False,
                "RESULT": "EXCLUDED"
            })

        # -------------------------------------------------------------
        # 3. [공정] 가동 범위 결측치/이상치 확인 (LOTID 단위)
        # -------------------------------------------------------------
        if lotid not in existing_lots:
            proc_pass = "FAIL"
            reason = "DB 내 수집 데이터 없음"
        elif lotid in invalid_lots:
            proc_pass = "FAIL"
            reason = "결측치 또는 이상치 포함됨"
        else:
            proc_pass = "PASS"
            reason = "모든 시계열 데이터 정상"
            
        proc_results.append({
            "LOTID": lotid,
            "EVAL_REASON": reason,
            "RESULT": proc_pass
        })

    # 데이터프레임 변환 및 CSV 저장
    df_img = pd.DataFrame(img_results)
    df_txt = pd.DataFrame(txt_results)
    df_proc = pd.DataFrame(proc_results)
    
    df_img.to_csv("./KOTCA_result/2_input_range_image.csv", index=False, encoding="utf-8-sig")
    df_txt.to_csv("./KOTCA_result/2_input_range_text.csv", index=False, encoding="utf-8-sig")
    df_proc.to_csv("./KOTCA_result/2_input_range_process.csv", index=False, encoding="utf-8-sig")
    
    # 비율 및 점수 계산
    img_score = (valid_pixel_count / total_rows) * 100
    txt_score = (valid_keyword_count / total_text_target * 100) if total_text_target > 0 else 100.0
    proc_score = (valid_proc_count / total_proc_rows * 100) if total_proc_rows > 0 else 0.0
    total_score = (img_score + txt_score + proc_score) / 3

    print("-> [완료] 2_input_range_image.csv, 2_input_range_text.csv, 2_input_range_process.csv 생성 완료\n")
    
    # 요약 및 기준 출력
    print("="*65)
    print(" [Step 2] 입력값 범위 정합성 검증 결과 리포트")
    print("="*65)
    
    print(f" [이미지] 정합성 점수: {img_score:>6.2f}%")
    print(f"   - 현황: PASS {len(df_img[df_img['RESULT']=='PASS'])}건 / FAIL {len(df_img[df_img['RESULT']=='FAIL'])}건")
    print("   - 기준: 이미지 파일이 물리적으로 존재하고 픽셀 배열(0~255)이 정상적으로 로드되는지 확인")
    print(f"   - 상세: 픽셀 배열 정상({valid_pixel_count}/{total_rows})\n")

    print(f" [텍스트] 정합성 점수: {txt_score:>6.2f}%")
    print(f"   - 현황: PASS {valid_keyword_count}건 / FAIL {total_text_target - valid_keyword_count}건 (검증 제외: {total_rows - total_text_target}건)")
    print("   - 기준: ERRORTYPE이 존재하는 실제 에러 데이터 중 지정된 사전 키워드(녹|찍힘|균열|스크레치) 포함 여부")
    print(f"   - 상세: 사전 키워드 포함({valid_keyword_count}/{total_text_target})\n")

    print(f" [공정]   정합성 점수: {proc_score:>6.2f}%")
    print(f"   - 현황: PASS {len(df_proc[df_proc['RESULT']=='PASS'])}건(LOTID 기준) / FAIL {len(df_proc[df_proc['RESULT']=='FAIL'])}건")
    print("   - 기준: PROC_TYPE별 필수 센서 데이터 누락(결측치) 및 정의된 가동 범위 이탈(이상치) 여부")
    print(f"   - 상세: 결측/이상치 통과({valid_proc_count}/{total_proc_rows})\n")
    
    print("-" * 65)
    print(f" >>> [입력값 범위] 전체 합산 정합성 평균 : {total_score:>6.2f}%")
    print("=" * 65 + "\n")

    return total_score

# 실행부
score_2 = run_step2_input_range_csv(df_validation, engine)

-> [Step 2] 입력값 범위 정합성 검증 및 CSV 추출을 시작합니다.
-> [완료] 2_input_range_image.csv, 2_input_range_text.csv, 2_input_range_process.csv 생성 완료

 [Step 2] 입력값 범위 정합성 검증 결과 리포트
 [이미지] 정합성 점수: 100.00%
   - 현황: PASS 11000건 / FAIL 0건
   - 기준: 이미지 파일이 물리적으로 존재하고 픽셀 배열(0~255)이 정상적으로 로드되는지 확인
   - 상세: 픽셀 배열 정상(11000/11000)

 [텍스트] 정합성 점수: 100.00%
   - 현황: PASS 4400건 / FAIL 0건 (검증 제외: 6600건)
   - 기준: ERRORTYPE이 존재하는 실제 에러 데이터 중 지정된 사전 키워드(녹|찍힘|균열|스크레치) 포함 여부
   - 상세: 사전 키워드 포함(4400/4400)

 [공정]   정합성 점수: 100.00%
   - 현황: PASS 11000건(LOTID 기준) / FAIL 0건
   - 기준: PROC_TYPE별 필수 센서 데이터 누락(결측치) 및 정의된 가동 범위 이탈(이상치) 여부
   - 상세: 결측/이상치 통과(1218627/1218627)

-----------------------------------------------------------------
 >>> [입력값 범위] 전체 합산 정합성 평균 : 100.00%



In [48]:
def run_step3_format_csv(df_validation, engine):
    print("-> [Step 3] 데이터 형식 정합성 검증 및 CSV 추출을 시작합니다.")
    
    img_results = []
    txt_results = []
    proc_results = []
    
    total_rows = len(df_validation)
    if total_rows == 0:
        print("[오류] 검증할 데이터프레임이 비어 있습니다.")
        return

    # 통계 집계용 변수
    valid_ext_count = 0
    valid_template_count = 0
    total_text_target = 0
    
    # [사전 준비] 공정 DB 스키마 1회 전역(Global) 검증
    schema_passed_count = 0
    expected_cols = {
        'DAID': 'object', 'LOTID': 'object', 'DATIME': 'datetime', 
        'PROC_TYPE': 'object', 'WELD_CURR': 'numeric', 'WELD_VOLT': 'numeric', 
        'WELD_TEMP': 'numeric', 'PAINT_PRESS': 'numeric', 'PAINT_TEMP': 'numeric', 
        'PAINT_HUMID': 'numeric', 'ISVALID': 'object', 'REMARK': 'object'
    }
    total_expected = len(expected_cols)
    
    try:
        df_proc_schema = pd.read_sql("SELECT * FROM ai_proc_davalue LIMIT 1", con=engine)
        schema_passed_count = sum(1 for col in expected_cols.keys() if col in df_proc_schema.columns)
    except Exception as e:
        print(f"[오류] DB 스키마 조회 실패: {e}")
        
    is_schema_valid = (schema_passed_count == total_expected)

    # 행별 전수 검사 루프
    for idx, row in df_validation.iterrows():
        lotid = row.get('LOTID')
        
        # -------------------------------------------------------------
        # 1. [이미지] 포맷 확장자 확인
        # -------------------------------------------------------------
        path = str(row.get('FILEPATH'))
        is_valid_ext = False
        
        if pd.notnull(row.get('FILEPATH')):
            if path.lower().endswith(('.jpg', '.png')):
                is_valid_ext = True
                valid_ext_count += 1
                
        img_pass = "PASS" if is_valid_ext else "FAIL"
        img_results.append({
            "LOTID": lotid,
            "FILEPATH": path,
            "VALID_EXTENSION": is_valid_ext,
            "RESULT": img_pass
        })

        # -------------------------------------------------------------
        # 2. [텍스트] 파인튜닝용 템플릿 포맷 확인
        # -------------------------------------------------------------
        etype = str(row.get('ERRORTYPE')).strip().upper()
        etext = str(row.get('ERRORTEXT')).strip()
        
        # ERRORTYPE이 존재하고 'NAN'이 아닌 경우만 검사
        is_valid_target = pd.notnull(row.get('ERRORTYPE')) and etype != 'NAN'
        
        if is_valid_target:
            total_text_target += 1
            if etype == 'NONE':
                pattern = re.compile(r'^부품명:\s*[^,]+,\s*품질상태:\s*[^,]+입니다\.$')
            else:
                pattern = re.compile(r'^부품명:\s*[^,]+,\s*품질상태:\s*[^,]+,\s*원인:\s*.+\s*입니다\.$')
            
            is_template_match = bool(pattern.match(etext))
            if is_template_match:
                valid_template_count += 1
                
            txt_pass = "PASS" if is_template_match else "FAIL"
            txt_results.append({
                "LOTID": lotid,
                "ERRORTYPE": etype,
                "ERRORTEXT": etext,
                "TEMPLATE_MATCH": is_template_match,
                "RESULT": txt_pass
            })
        else:
            txt_results.append({
                "LOTID": lotid,
                "ERRORTYPE": etype,
                "ERRORTEXT": etext,
                "TEMPLATE_MATCH": False,
                "RESULT": "EXCLUDED"
            })

        # -------------------------------------------------------------
        # 3. [공정] 데이터프레임 스키마 확인
        # -------------------------------------------------------------
        # 전역 스키마 검증 결과를 개별 행에 매핑
        proc_pass = "PASS" if is_schema_valid else "FAIL"
        proc_results.append({
            "LOTID": lotid,
            "SCHEMA_PASSED_COLS": f"{schema_passed_count}/{total_expected}",
            "RESULT": proc_pass
        })

    # 데이터프레임 변환 및 CSV 저장
    df_img = pd.DataFrame(img_results)
    df_txt = pd.DataFrame(txt_results)
    df_proc = pd.DataFrame(proc_results)
    
    df_img.to_csv("./KOTCA_result/3_format_image.csv", index=False, encoding="utf-8-sig")
    df_txt.to_csv("./KOTCA_result/3_format_text.csv", index=False, encoding="utf-8-sig")
    df_proc.to_csv("./KOTCA_result/3_format_process.csv", index=False, encoding="utf-8-sig")
    
    # 비율 및 점수 계산
    img_score = (valid_ext_count / total_rows) * 100
    txt_score = (valid_template_count / total_text_target * 100) if total_text_target > 0 else 100.0
    proc_score = (schema_passed_count / total_expected * 100) if total_expected > 0 else 0.0
    total_score = (img_score + txt_score + proc_score) / 3

    print("-> [완료] 3_format_image.csv, 3_format_text.csv, 3_format_process.csv 생성 완료\n")
    
    # 요약 및 기준 출력
    print("="*65)
    print(" [Step 3] 데이터 형식 정합성 검증 결과 리포트")
    print("="*65)
    
    print(f" [이미지] 정합성 점수: {img_score:>6.2f}%")
    print(f"   - 현황: PASS {len(df_img[df_img['RESULT']=='PASS'])}건 / FAIL {len(df_img[df_img['RESULT']=='FAIL'])}건")
    print("   - 기준: 이미지 파일 포맷이 규정된 확장자(.jpg, .png)와 정확히 일치하는지 확인")
    print(f"   - 상세: 확장자 검증 통과({valid_ext_count}/{total_rows})\n")

    print(f" [텍스트] 정합성 점수: {txt_score:>6.2f}%")
    print(f"   - 현황: PASS {valid_template_count}건 / FAIL {total_text_target - valid_template_count}건 (검증 제외: {total_rows - total_text_target}건)")
    print("   - 기준: ERRORTYPE 유무(정상/에러)에 따른 VLM 파인튜닝용 템플릿 정규식 매칭 여부")
    print(f"   - 상세: 파인튜닝 템플릿 포맷 일치({valid_template_count}/{total_text_target})\n")

    print(f" [공정]   정합성 점수: {proc_score:>6.2f}%")
    print(f"   - 현황: PASS {len(df_proc[df_proc['RESULT']=='PASS'])}건(LOTID 기준) / FAIL {len(df_proc[df_proc['RESULT']=='FAIL'])}건")
    print("   - 기준: 공정 DB(ai_proc_davalue) 테이블 내 필수 센서/메타 데이터 12개 컬럼 존재 여부")
    print(f"   - 상세: 스키마 검증({schema_passed_count}/{total_expected} 필수 컬럼)\n")
    
    print("-" * 65)
    print(f" >>> [데이터 형식] 전체 합산 정합성 평균 : {total_score:>6.2f}%")
    print("=" * 65 + "\n")

    return total_score

# 실행부
score_3 = run_step3_format_csv(df_validation, engine)

-> [Step 3] 데이터 형식 정합성 검증 및 CSV 추출을 시작합니다.
-> [완료] 3_format_image.csv, 3_format_text.csv, 3_format_process.csv 생성 완료

 [Step 3] 데이터 형식 정합성 검증 결과 리포트
 [이미지] 정합성 점수: 100.00%
   - 현황: PASS 11000건 / FAIL 0건
   - 기준: 이미지 파일 포맷이 규정된 확장자(.jpg, .png)와 정확히 일치하는지 확인
   - 상세: 확장자 검증 통과(11000/11000)

 [텍스트] 정합성 점수: 100.00%
   - 현황: PASS 5500건 / FAIL 0건 (검증 제외: 5500건)
   - 기준: ERRORTYPE 유무(정상/에러)에 따른 VLM 파인튜닝용 템플릿 정규식 매칭 여부
   - 상세: 파인튜닝 템플릿 포맷 일치(5500/5500)

 [공정]   정합성 점수: 100.00%
   - 현황: PASS 11000건(LOTID 기준) / FAIL 0건
   - 기준: 공정 DB(ai_proc_davalue) 테이블 내 필수 센서/메타 데이터 12개 컬럼 존재 여부
   - 상세: 스키마 검증(12/12 필수 컬럼)

-----------------------------------------------------------------
 >>> [데이터 형식] 전체 합산 정합성 평균 : 100.00%



In [49]:
def run_step4_syntax_csv(df_validation, engine):
    print("-> [Step 4] 구문 정확성 검증 및 CSV 추출을 시작합니다.")
    
    img_results = []
    txt_results = []
    proc_results = []
    
    total_rows = len(df_validation)
    if total_rows == 0:
        print("[오류] 검증할 데이터프레임이 비어 있습니다.")
        return

    # -------------------------------------------------------------
    # [사전 준비] 공정 DB 수집 누락(결측치) 일괄 판별
    # -------------------------------------------------------------
    validation_lots = df_validation['LOTID'].dropna().unique().tolist()
    total_val_lots = len(validation_lots)
    
    existing_lots = set()
    invalid_lots = set()
    total_proc_rows = 0
    valid_proc_count = 0
    
    if total_val_lots > 0:
        format_strings = ','.join(['%s'] * total_val_lots)
        try:
            query = f"SELECT DAID, lotid, DATIME, PROC_TYPE, WELD_CURR, WELD_VOLT, WELD_TEMP, PAINT_PRESS, PAINT_TEMP, PAINT_HUMID FROM ai_proc_davalue WHERE lotid IN ({format_strings})"
            df_proc = pd.read_sql(query, con=engine, params=tuple(validation_lots))
            total_proc_rows = len(df_proc)
            
            if total_proc_rows > 0:
                existing_lots = set(df_proc['lotid'].dropna().unique())
                
                # 공통 필수 컬럼 및 공정별 필수 센서 누락 확인
                cond_general_miss = df_proc[['DAID', 'lotid', 'DATIME', 'PROC_TYPE']].isnull().any(axis=1)
                cond_weld_miss = (df_proc['PROC_TYPE'] == '용접') & df_proc[['WELD_CURR', 'WELD_VOLT', 'WELD_TEMP']].isnull().any(axis=1)
                cond_paint_miss = (df_proc['PROC_TYPE'] == '도장') & df_proc[['PAINT_PRESS', 'PAINT_TEMP', 'PAINT_HUMID']].isnull().any(axis=1)
                
                is_missing = cond_general_miss | cond_weld_miss | cond_paint_miss
                valid_proc_count = (~is_missing).sum()
                
                df_proc['IS_MISSING'] = is_missing
                invalid_lots = set(df_proc[df_proc['IS_MISSING']]['lotid'].dropna().unique())
                
        except Exception as e:
            print(f"[오류] DB 조회 실패: {e}")

    # 통계 집계용 변수
    valid_img_count = 0
    vision_missing_count = 0
    valid_text_count = 0
    total_text_target = 0
    
    vision_cols = [c for c in ['LOTID', 'FILEPATH', 'DATIME'] if c in df_validation.columns]
    broken_pattern = re.compile(r'[\ufffd]|[ㄱ-ㅎㅏ-ㅣ]')

    # 행별 전수 검사 루프
    for idx, row in df_validation.iterrows():
        lotid = row.get('LOTID')
        
        # -------------------------------------------------------------
        # 1. [이미지] 디코딩 검증 및 비전 메타데이터 누락 확인
        # -------------------------------------------------------------
        path = str(row.get('FILEPATH'))
        is_decoded = False
        
        # 이미지 디코딩 확인
        if os.path.exists(path):
            img = cv2.imread(path)
            if img is not None:
                is_decoded = True
                valid_img_count += 1
                
        # 비전 컬럼(LOTID, FILEPATH, DATIME) 누락 확인
        is_vision_valid = not row[vision_cols].isnull().any()
        if is_vision_valid:
            vision_missing_count += 1
            
        img_pass = "PASS" if (is_decoded and is_vision_valid) else "FAIL"
        img_results.append({
            "LOTID": lotid,
            "FILEPATH": path,
            "DECODING_SUCCESS": is_decoded,
            "VISION_COLS_VALID": is_vision_valid,
            "RESULT": img_pass
        })

        # -------------------------------------------------------------
        # 2. [텍스트] 깨진 문자열 확인
        # -------------------------------------------------------------
        raw_err_text = row.get('ERRORTEXT')
        is_text_empty = pd.isnull(raw_err_text) or str(raw_err_text).strip() == "" or str(raw_err_text).strip().lower() in ['nan', 'none']
        
        if not is_text_empty:
            total_text_target += 1
            etext = str(raw_err_text)
            has_broken = bool(broken_pattern.search(etext))
            
            if not has_broken:
                valid_text_count += 1
                txt_pass = "PASS"
            else:
                txt_pass = "FAIL"
                
            txt_results.append({
                "LOTID": lotid,
                "ERRORTEXT": etext,
                "HAS_BROKEN_CHARS": has_broken,
                "RESULT": txt_pass
            })
        else:
            txt_results.append({
                "LOTID": lotid,
                "ERRORTEXT": raw_err_text,
                "HAS_BROKEN_CHARS": False,
                "RESULT": "EXCLUDED"
            })

        # -------------------------------------------------------------
        # 3. [공정] 수집 누락(결측치) 확인
        # -------------------------------------------------------------
        if pd.isnull(lotid) or lotid not in existing_lots:
            proc_pass = "FAIL"
            reason = "DB 내 수집 데이터 없음"
        elif lotid in invalid_lots:
            proc_pass = "FAIL"
            reason = "필수 컬럼 누락(결측치) 존재"
        else:
            proc_pass = "PASS"
            reason = "모든 필수 데이터 누락 없음"
            
        proc_results.append({
            "LOTID": lotid,
            "EVAL_REASON": reason,
            "RESULT": proc_pass
        })

    # 데이터프레임 변환 및 CSV 저장
    df_img = pd.DataFrame(img_results)
    df_txt = pd.DataFrame(txt_results)
    df_proc = pd.DataFrame(proc_results)
    
    df_img.to_csv("./KOTCA_result/4_syntax_image.csv", index=False, encoding="utf-8-sig")
    df_txt.to_csv("./KOTCA_result/4_syntax_text.csv", index=False, encoding="utf-8-sig")
    df_proc.to_csv("./KOTCA_result/4_syntax_process.csv", index=False, encoding="utf-8-sig")
    
    # 비율 및 점수 계산
    img_score = (((valid_img_count / total_rows) * 100) + ((vision_missing_count / total_rows) * 100)) / 2
    txt_score = (valid_text_count / total_text_target * 100) if total_text_target > 0 else 100.0
    proc_score = (valid_proc_count / total_proc_rows * 100) if total_proc_rows > 0 else 0.0
    total_score = (img_score + txt_score + proc_score) / 3

    print("-> [완료] 4_syntax_image.csv, 4_syntax_text.csv, 4_syntax_process.csv 생성 완료\n")
    
    # 요약 및 기준 출력
    print("="*65)
    print(" [Step 4] 구문 정확성 평가 결과 리포트")
    print("="*65)
    
    print(f" [이미지] 정합성 점수: {img_score:>6.2f}%")
    print(f"   - 현황: PASS {len(df_img[df_img['RESULT']=='PASS'])}건 / FAIL {len(df_img[df_img['RESULT']=='FAIL'])}건")
    print("   - 기준: cv2 디코딩 손상 여부 및 비전 테이블의 필수 컬럼(LOTID, FILEPATH, DATIME) 누락 점검")
    print(f"   - 상세: 디코딩({valid_img_count}/{total_rows}) / 비전 컬럼({vision_missing_count}/{total_rows})\n")

    print(f" [텍스트] 정합성 점수: {txt_score:>6.2f}%")
    print(f"   - 현황: PASS {valid_text_count}건 / FAIL {total_text_target - valid_text_count}건 (검증 제외: {total_rows - total_text_target}건)")
    print("   - 기준: ERRORTEXT 내 유니코드 깨짐 문자(\\ufffd) 및 비정상 단일 자음/모음 존재 여부")
    print(f"   - 상세: 깨짐 문자 없음({valid_text_count}/{total_text_target})\n")

    print(f" [공정]   정합성 점수: {proc_score:>6.2f}%")
    print(f"   - 현황: PASS {len(df_proc[df_proc['RESULT']=='PASS'])}건(LOTID 기준) / FAIL {len(df_proc[df_proc['RESULT']=='FAIL'])}건")
    print("   - 기준: 공정 데이터 (PROC_TYPE별) 필수 센서 데이터 수집 누락(결측치) 검증")
    print(f"   - 상세: 센서 데이터 수집 통과({valid_proc_count}/{total_proc_rows})\n")
    
    print("-" * 65)
    print(f" >>> [구문 정확성] 전체 합산 정합성 평균 : {total_score:>6.2f}%")
    print("=" * 65 + "\n")

    return total_score

# 실행부
score_4 = run_step4_syntax_csv(df_validation, engine)

-> [Step 4] 구문 정확성 검증 및 CSV 추출을 시작합니다.
-> [완료] 4_syntax_image.csv, 4_syntax_text.csv, 4_syntax_process.csv 생성 완료

 [Step 4] 구문 정확성 평가 결과 리포트
 [이미지] 정합성 점수: 100.00%
   - 현황: PASS 11000건 / FAIL 0건
   - 기준: cv2 디코딩 손상 여부 및 비전 테이블의 필수 컬럼(LOTID, FILEPATH, DATIME) 누락 점검
   - 상세: 디코딩(11000/11000) / 비전 컬럼(11000/11000)

 [텍스트] 정합성 점수: 100.00%
   - 현황: PASS 5500건 / FAIL 0건 (검증 제외: 5500건)
   - 기준: ERRORTEXT 내 유니코드 깨짐 문자(\ufffd) 및 비정상 단일 자음/모음 존재 여부
   - 상세: 깨짐 문자 없음(5500/5500)

 [공정]   정합성 점수: 100.00%
   - 현황: PASS 11000건(LOTID 기준) / FAIL 0건
   - 기준: 공정 데이터 (PROC_TYPE별) 필수 센서 데이터 수집 누락(결측치) 검증
   - 상세: 센서 데이터 수집 통과(1218627/1218627)

-----------------------------------------------------------------
 >>> [구문 정확성] 전체 합산 정합성 평균 : 100.00%



In [5]:
def run_step5_diversity_csv(df_validation, engine):
    print("-> [Step 5] 통계적 다양성 검증 및 CSV(특성 추출)를 시작합니다.")
    
    img_results = []
    txt_results = []
    proc_results = []
    
    total_rows = len(df_validation)
    if total_rows == 0:
        print("[오류] 검증할 데이터프레임이 비어 있습니다.")
        return

    # -------------------------------------------------------------
    # [사전 준비] 공정 DB(ai_proc_prevalue) 정상/불량 상태 일괄 로드
    # -------------------------------------------------------------
    null_iserror_lots = df_validation[df_validation['ISERROR'].isnull()]['LOTID'].dropna().unique().tolist()
    prevalue_dict = {}
    
    if null_iserror_lots:
        format_strings = ','.join(['%s'] * len(null_iserror_lots))
        try:
            query_pre = f"SELECT lotid, ISERROR FROM ai_proc_prevalue WHERE lotid IN ({format_strings})"
            df_pre = pd.read_sql(query_pre, con=engine, params=tuple(null_iserror_lots))
            df_pre = df_pre.drop_duplicates(subset=['lotid'], keep='last')
            prevalue_dict = dict(zip(df_pre['lotid'], df_pre['ISERROR']))
        except Exception as e:
            print(f"[오류] ai_proc_prevalue DB 조회 실패: {e}")

    def classify_status(val):
        if pd.isnull(val): return 'UNKNOWN'
        return 'NORMAL' if str(val).strip().upper() in ['정상'] else 'DEFECT'

    # 행별 특성 추출 루프
    for idx, row in df_validation.iterrows():
        lotid = row.get('LOTID')
        
        # -------------------------------------------------------------
        # 1. [이미지] 밝기 및 대비 특성 추출
        # -------------------------------------------------------------
        path = str(row.get('FILEPATH'))
        b_val, c_val = None, None
        
        if os.path.exists(path):
            img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
            if img is not None:
                m, s = cv2.meanStdDev(img)
                b_val = m[0][0]
                c_val = s[0][0]
                
        img_results.append({
            "LOTID": lotid,
            "FILEPATH": path,
            "BRIGHTNESS": b_val,
            "CONTRAST": c_val
        })

        # -------------------------------------------------------------
        # 2. [텍스트] 불량 유형 및 문장 길이 추출
        # -------------------------------------------------------------
        etype = str(row.get('ERRORTYPE')).strip().upper()
        raw_etext = row.get('ERRORTEXT')
        etext = str(raw_etext).strip() if pd.notnull(raw_etext) else ""
        
        has_text = (etext != "") and (etext.upper() not in ['NAN', 'NONE', 'NULL'])
        
        if has_text:
            txt_results.append({
                "LOTID": lotid,
                "ERRORTYPE": etype,
                "ERRORTEXT": etext,
                "TEXT_LENGTH": len(etext)
            })
        else:
            # 텍스트 검증에서 제외되는 데이터
            txt_results.append({
                "LOTID": lotid,
                "ERRORTYPE": etype,
                "ERRORTEXT": etext,
                "TEXT_LENGTH": "EXCLUDED"
            })

        # -------------------------------------------------------------
        # 3. [공정] 정상/불량 상태 분류 추출
        # -------------------------------------------------------------
        iserror_val = row.get('ISERROR')
        data_source = "ai_vision_davalue"
        
        if pd.isnull(iserror_val):
            iserror_val = prevalue_dict.get(lotid, None)
            data_source = "ai_proc_prevalue" if iserror_val is not None else "NOT_FOUND"
            
        final_status = classify_status(iserror_val)
        
        proc_results.append({
            "LOTID": lotid,
            "RAW_ISERROR": iserror_val,
            "DATA_SOURCE": data_source,
            "CLASSIFIED_STATUS": final_status
        })

    # 데이터프레임 변환 및 CSV 저장
    df_img = pd.DataFrame(img_results)
    df_txt = pd.DataFrame(txt_results)
    df_proc = pd.DataFrame(proc_results)
    
    df_img.to_csv("./KOTCA_result/5_diversity_image.csv", index=False, encoding="utf-8-sig")
    df_txt.to_csv("./KOTCA_result/5_diversity_text.csv", index=False, encoding="utf-8-sig")
    df_proc.to_csv("./KOTCA_result/5_diversity_process.csv", index=False, encoding="utf-8-sig")
    
    print("-> [완료] 5_diversity_image.csv, 5_diversity_text.csv, 5_diversity_process.csv 생성 완료\n")
    
    # -------------------------------------------------------------
    # [통계 집계 및 리포트 출력]
    # -------------------------------------------------------------
    print("="*65)
    print(" [Step 5] 통계적 다양성 평가 결과 리포트")
    print("="*65)

    # 1. 이미지 통계
    b_list = df_img['BRIGHTNESS'].dropna().tolist()
    c_list = df_img['CONTRAST'].dropna().tolist()
    
    img_score = 50.0
    img_msg = "데이터 부족"
    if b_list and c_list:
        b_std, c_std = np.std(b_list), np.std(c_list)
        img_score = 100.0 if (b_std > 5.0 and c_std > 10.0) else 50.0
        pass_fail = "합격" if img_score == 100.0 else "불합격"
        img_msg = f"편차(밝기:{b_std:.1f}, 대비:{c_std:.1f}) -> {pass_fail} [기준: 밝기>5.0, 대비>10.0]"
        
    print(f" [이미지] 정합성 점수: {img_score:>6.2f}%")
    print(f"   - 결과: {img_msg}")
    print(f"   - 산출: {len(b_list)}건의 이미지 분석\n")

    # 2. 텍스트 통계
    txt_score = 0.0
    txt_msg = "불량 데이터 부족"
    df_txt_valid = df_txt[df_txt['TEXT_LENGTH'] != "EXCLUDED"].copy()
    if not df_txt_valid.empty:
        df_txt_valid['TEXT_LENGTH'] = pd.to_numeric(df_txt_valid['TEXT_LENGTH'])
        mean_len = df_txt_valid.groupby('ERRORTYPE')['TEXT_LENGTH'].mean()
        
        if len(mean_len) > 0:
            max_diff = mean_len.max() - mean_len.min()
            txt_score = 100.0 if max_diff <= 20.0 else 50.0
            pass_fail = "합격" if txt_score == 100.0 else "불합격"
            txt_msg = f"유형별 최대 길이차:{max_diff:.1f}자 -> {pass_fail} [기준: 20자 이내]"
            
    print(f" [텍스트] 정합성 점수: {txt_score:>6.2f}%")
    print(f"   - 결과: {txt_msg}")
    print(f"   - 산출: {len(df_txt_valid)}건의 텍스트 길이 분석\n")

    # 3. 공정(비율) 통계
    n_cnt = (df_proc['CLASSIFIED_STATUS'] == 'NORMAL').sum()
    d_cnt = (df_proc['CLASSIFIED_STATUS'] == 'DEFECT').sum()
    valid_counts = n_cnt + d_cnt
    
    proc_score = 50.0
    proc_msg = "분류 데이터 없음"
    if valid_counts > 0:
        defect_ratio = (d_cnt / valid_counts) * 100
        proc_score = 100.0 if 20.0 <= defect_ratio <= 70.0 else 50.0
        pass_fail = "합격" if proc_score == 100.0 else "불합격"
        proc_msg = f"정상 {n_cnt}건, 불량 {d_cnt}건 (불량률:{defect_ratio:.1f}%) -> {pass_fail} [기준: 20~70%]"

    print(f" [공정]   정합성 점수: {proc_score:>6.2f}%")
    print(f"   - 결과: {proc_msg}")
    print(f"   - 산출: 총 {valid_counts}건의 상태 분류\n")

    total_score = (img_score + txt_score + proc_score) / 3
    print("-" * 65)
    print(f" >>> [통계적 다양성] 전체 합산 정합성 평균 : {total_score:>6.2f}%")
    print("=" * 65 + "\n")

    return total_score
# 실행부
score_5 = run_step5_diversity_csv(df_validation, engine)

-> [Step 5] 통계적 다양성 검증 및 CSV(특성 추출)를 시작합니다.
-> [완료] 5_diversity_image.csv, 5_diversity_text.csv, 5_diversity_process.csv 생성 완료

 [Step 5] 통계적 다양성 평가 결과 리포트
 [이미지] 정합성 점수: 100.00%
   - 결과: 편차(밝기:6.8, 대비:15.2) -> 합격 [기준: 밝기>5.0, 대비>10.0]
   - 산출: 11000건의 이미지 분석

 [텍스트] 정합성 점수: 100.00%
   - 결과: 유형별 최대 길이차:10.1자 -> 합격 [기준: 20자 이내]
   - 산출: 5500건의 텍스트 길이 분석

 [공정]   정합성 점수: 100.00%
   - 결과: 정상 4928건, 불량 6072건 (불량률:55.2%) -> 합격 [기준: 20~70%]
   - 산출: 총 11000건의 상태 분류

-----------------------------------------------------------------
 >>> [통계적 다양성] 전체 합산 정합성 평균 : 100.00%



In [53]:
from datetime import datetime

def generate_final_report(score_dict, output_dir="."):
    """
    [최종 통합 리포트 생성]
    1~5단계의 정합성 점수를 취합하여 평균을 내고, 
    지정된 프로젝트 폴더에 1장짜리 txt 파일로 저장합니다.
    """
    print("-> [Step 6] 최종 통합 정합성 평가 및 리포트 생성을 시작합니다.")
    
    if not score_dict:
        print("[오류] 평가 점수 데이터가 없습니다.")
        return

    # 종합 평균 계산
    total_sum = sum(score_dict.values())
    avg_score = total_sum / len(score_dict)
    
    # 시험 규격(99%) 기준 PASS/FAIL 판별
    final_status = "PASS" if avg_score >= 99.0 else "FAIL"
    current_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    # 리포트 텍스트 포맷팅
    report_lines = [
        "=" * 70,
        " [학습 데이터 정합성 평가 최종 리포트]",
        "=" * 70,
        f" 평가 완료 일시: {current_time}",
        "-" * 70,
        " [세부 평가 항목별 점수]"
    ]
    
    for step_name, score in score_dict.items():
        report_lines.append(f"   - {step_name:<18} : {score:>6.2f}%")
        
    report_lines.extend([
        "-" * 70,
        f" >>> 종합 정합성 평균 : {avg_score:>6.2f}%",
        f" >>> 최종 시험 판정   : {final_status} (규격 기준: 99.0% 이상)",
        "=" * 70
    ])
    
    report_text = "\n".join(report_lines)
    
    # 텍스트 파일로 저장
    file_path = os.path.join(output_dir, "final_report.txt")
    try:
        with open(file_path, "w", encoding="utf-8") as f:
            f.write(report_text)
        print(f"-> [완료] 최종 요약 리포트 저장 성공: {file_path}\n")
    except Exception as e:
        print(f"[오류] 파일 저장 중 문제 발생: {e}\n")
        
    # 터미널에 동일한 리포트 출력
    print(report_text)

# 2. 딕셔너리로 취합
final_scores = {
    "데이터 구조 정합성": score_1,
    "입력값 범위 정합성": score_2,
    "데이터 형식 정합성": score_3,
    "구문 정확성": score_4,
    "통계적 다양성": score_5
}

# 3. 현재 작업 중인 프로젝트 폴더 경로 (예: './project_A_logs')
project_path = "./KOTCA_result" 

# 4. 최종 리포트 생성
generate_final_report(final_scores, output_dir=project_path)

-> [Step 6] 최종 통합 정합성 평가 및 리포트 생성을 시작합니다.
-> [완료] 최종 요약 리포트 저장 성공: ./KOTCA_result\final_report.txt

 [학습 데이터 정합성 평가 최종 리포트]
 평가 완료 일시: 2026-06-08 14:33:09
----------------------------------------------------------------------
 [세부 평가 항목별 점수]
   - 데이터 구조 정합성         : 100.00%
   - 입력값 범위 정합성         : 100.00%
   - 데이터 형식 정합성         : 100.00%
   - 구문 정확성             : 100.00%
   - 통계적 다양성            : 100.00%
----------------------------------------------------------------------
 >>> 종합 정합성 평균 : 100.00%
 >>> 최종 시험 판정   : PASS (규격 기준: 99.0% 이상)


In [52]:
class MultimodalDataValidator:
    def __init__(self, df_validation, engine):
        self.df = df_validation
        self.engine = engine
        self.total_rows = len(df_validation)
        self.scores = {}
        
        # 반복적인 DB 조회를 최소화하기 위해 공통 고유 LOTID 추출
        self.validation_lots = self.df['LOTID'].dropna().unique().tolist()
        self.total_val_lots = len(self.validation_lots)

    def run_all_validations(self):
        """5개의 정합성 평가를 순차적으로 조용히 실행하고 세부 결과가 포함된 최종 리포트를 출력합니다."""
        if self.total_rows == 0:
            print("[오류] 검증할 데이터프레임이 비어 있습니다.")
            return
            
        print("-> [진행 중] 데이터 통합 정합성 평가 파이프라인 가동")
        
        self.scores['structure'] = self._validate_structure()
        self.scores['input_range'] = self._validate_input_range()
        self.scores['format'] = self._validate_format()
        self.scores['syntax'] = self._validate_syntax()
        self.scores['diversity'] = self._validate_diversity()
        
        self._print_final_report()

    # -------------------------------------------------------------
    # 1. 데이터 구조 정합성 평가
    # -------------------------------------------------------------
    def _validate_structure(self):
        # [이미지] filepath 기준 파일명 정형화 검증 및 물리 파일 존재 확인
        
        valid_filename_count = 0
        physical_exist_count = 0
        valid_extensions = ['.jpg', '.jpeg', '.png']
        
        for idx, row in self.df.iterrows():
            path = str(row['FILEPATH'])
            actual_stem, ext = os.path.splitext(os.path.basename(path))
            
            if os.path.exists(path):
                physical_exist_count += 1

            raw_datime = row.get('DATIME')
            lotid = row.get('LOTID')
            if pd.notnull(raw_datime) and pd.notnull(lotid):
                try:
                    formatted_time = pd.to_datetime(raw_datime).strftime('%Y%m%d%H%M%S')
                    if (actual_stem == f"{formatted_time}_{lotid}") and (ext.lower() in valid_extensions):
                        valid_filename_count += 1
                except: pass

        img_score = (((valid_filename_count / self.total_rows) * 100) + ((physical_exist_count / self.total_rows) * 100)) / 2
        img_msg = f"파일명({valid_filename_count}/{self.total_rows}) / 물리파일({physical_exist_count}/{self.total_rows})"

        # [텍스트] ISERROR가 null일 때 ERRORTEXT도 비어있는지 조건부 매칭 검증
        null_iserror = self.df['ISERROR'].isnull()
        text_is_empty = self.df['ERRORTEXT'].isnull() | (self.df['ERRORTEXT'].astype(str).str.strip() == "")
        valid_text_count = ((null_iserror & text_is_empty) | (~null_iserror & ~text_is_empty)).sum()
        txt_score = (valid_text_count / self.total_rows) * 100
        txt_msg = f"에러 매칭({valid_text_count}/{self.total_rows})"

        # [공정] LOTID를 필터링하여 ai_proc_davalue와 1:1 매칭 검증
        proc_distinct_lots = set()
        if self.total_val_lots > 0:
            format_strings = ','.join(['%s'] * self.total_val_lots)
            try:
                df_proc = pd.read_sql(f"SELECT DISTINCT lotid FROM ai_proc_davalue WHERE lotid IN ({format_strings})", con=self.engine, params=tuple(self.validation_lots))
                proc_distinct_lots = set(df_proc['lotid'].dropna().unique())
            except: pass
        
        matched_lot_count = len(set(self.validation_lots).intersection(proc_distinct_lots))
        proc_score = (matched_lot_count / self.total_val_lots * 100) if self.total_val_lots > 0 else 0.0
        proc_msg = f"LOTID 매칭({matched_lot_count}/{self.total_val_lots})"

        return {'img': img_score, 'img_msg': img_msg, 'txt': txt_score, 'txt_msg': txt_msg, 'proc': proc_score, 'proc_msg': proc_msg, 'total': (img_score + txt_score + proc_score) / 3}

    # -------------------------------------------------------------
    # 2. 입력값 범위 정합성 평가
    # -------------------------------------------------------------
    def _validate_input_range(self):
        # [이미지] 이미지 픽셀 값 정상 로드 및 픽셀 배열(0~255) 확인
        valid_pixel_count = sum(1 for p in self.df['FILEPATH'] if os.path.exists(str(p)) and cv2.imread(str(p)) is not None)
        img_score = (valid_pixel_count / self.total_rows) * 100
        img_msg = f"픽셀 배열 정상({valid_pixel_count}/{self.total_rows})"

        # [텍스트] ERRORTYPE이 존재하는 실제 에러 데이터(NONE 제외) 중 키워드 포함 여부 확인
        cond_has_text = self.df['ERRORTEXT'].notnull() & (self.df['ERRORTEXT'].astype(str).str.strip() != "")
        cond_real_error = self.df['ERRORTYPE'].notnull() & (self.df['ERRORTYPE'].astype(str).str.upper() != 'NONE') if 'ERRORTYPE' in self.df.columns else True
        df_text_target = self.df[cond_has_text & cond_real_error]
        valid_keyword_count = df_text_target['ERRORTEXT'].astype(str).str.contains('녹|찍힘|균열|스크레치', na=False).sum()
        txt_score = (valid_keyword_count / len(df_text_target) * 100) if len(df_text_target) > 0 else 100.0
        txt_msg = f"사전 키워드 포함({valid_keyword_count}/{len(df_text_target)})"

        # [공정] PROC_TYPE별 필수 컬럼 결측치 확인 및 지정된 범위를 벗어나는 이상치 검증
        proc_score = 0.0
        proc_msg = "검증 대상 없음"
        if self.total_val_lots > 0:
            format_strings = ','.join(['%s'] * self.total_val_lots)
            try:
                df_proc = pd.read_sql(f"SELECT lotid, PROC_TYPE, WELD_CURR, WELD_VOLT, WELD_TEMP, PAINT_PRESS, PAINT_TEMP, PAINT_HUMID FROM ai_proc_davalue WHERE lotid IN ({format_strings})", con=self.engine, params=tuple(self.validation_lots))
                
                cond_weld_missing = (df_proc['PROC_TYPE'] == '용접') & df_proc[['WELD_CURR', 'WELD_VOLT', 'WELD_TEMP']].isnull().any(axis=1)
                cond_paint_missing = (df_proc['PROC_TYPE'] == '도장') & df_proc[['PAINT_PRESS', 'PAINT_TEMP', 'PAINT_HUMID']].isnull().any(axis=1)
                
                # [확실하지 않음] 임시 이상치 범위이므로 수정 필요
                cond_weld_outlier = (df_proc['PROC_TYPE'] == '용접') & (((df_proc['WELD_CURR'] < 250) | (df_proc['WELD_CURR'] > 350)) | ((df_proc['WELD_VOLT'] < 24) | (df_proc['WELD_VOLT'] > 34)) | ((df_proc['WELD_TEMP'] < 100) | (df_proc['WELD_TEMP'] > 150)))
                cond_paint_outlier = (df_proc['PROC_TYPE'] == '도장') & (((df_proc['PAINT_PRESS'] < 2) | (df_proc['PAINT_PRESS'] > 6)) | ((df_proc['PAINT_TEMP'] < 20) | (df_proc['PAINT_TEMP'] > 30)) | ((df_proc['PAINT_HUMID'] < 40) | (df_proc['PAINT_HUMID'] > 70)))
                
                valid_proc_count = (~(cond_weld_missing | cond_paint_missing | cond_weld_outlier | cond_paint_outlier)).sum()
                proc_score = (valid_proc_count / len(df_proc) * 100) if len(df_proc) > 0 else 0.0
                proc_msg = f"결측/이상치 통과({valid_proc_count}/{len(df_proc)})"
            except: pass

        return {'img': img_score, 'img_msg': img_msg, 'txt': txt_score, 'txt_msg': txt_msg, 'proc': proc_score, 'proc_msg': proc_msg, 'total': (img_score + txt_score + proc_score) / 3}

    # -------------------------------------------------------------
    # 3. 데이터 형식 정합성 평가
    # -------------------------------------------------------------
    def _validate_format(self):
        # [이미지] 포맷 확장자 .jpg, .png 확장자만 정상으로 판별
        valid_ext_count = self.df['FILEPATH'].apply(lambda p: str(p).lower().endswith(('.jpg', '.png')) if pd.notnull(p) else False).sum()
        img_score = (valid_ext_count / self.total_rows) * 100
        img_msg = f"확장자 검증(.jpg/.png) ({valid_ext_count}/{self.total_rows})"

        # [텍스트] ERRORTYPE 유무에 따른 VLM 텍스트 포맷 일치 여부 판별
        df_template_target = self.df[self.df['ERRORTYPE'].notnull() & (self.df['ERRORTYPE'].astype(str).str.upper() != 'NAN')] if 'ERRORTYPE' in self.df.columns else pd.DataFrame()
        valid_template_count = 0
        for _, row in df_template_target.iterrows():
            etype, text = str(row['ERRORTYPE']).strip().upper(), str(row['ERRORTEXT']).strip()
            pattern = re.compile(r'^부품명:\s*[^,]+,\s*품질상태:\s*[^,]+입니다\.$') if etype == 'NONE' else re.compile(r'^부품명:\s*[^,]+,\s*품질상태:\s*[^,]+,\s*원인:\s*.+\s*입니다\.$')
            if pattern.match(text): valid_template_count += 1
                
        txt_score = (valid_template_count / len(df_template_target) * 100) if len(df_template_target) > 0 else 100.0
        txt_msg = f"파인튜닝 템플릿 포맷({valid_template_count}/{len(df_template_target)})"

        # [공정] 공정 데이터프레임(ai_proc_davalue)의 12개 컬럼 존재 여부 및 Dtype 검증
        proc_score = 0.0
        proc_msg = "DB 조회 불가"
        if self.total_val_lots > 0:
            try:
                df_proc_schema = pd.read_sql("SELECT * FROM ai_proc_davalue LIMIT 1", con=self.engine)
                expected = {'DAID': 'object', 'LOTID': 'object', 'DATIME': 'datetime', 'PROC_TYPE': 'object', 'WELD_CURR': 'numeric', 'WELD_VOLT': 'numeric', 'WELD_TEMP': 'numeric', 'PAINT_PRESS': 'numeric', 'PAINT_TEMP': 'numeric', 'PAINT_HUMID': 'numeric', 'ISVALID': 'object', 'REMARK': 'object'}
                passed = sum(1 for col in expected.keys() if col in df_proc_schema.columns)
                proc_score = (passed / len(expected)) * 100
                proc_msg = f"스키마 검증({passed}/{len(expected)} 필수 컬럼)"
            except: pass

        return {'img': img_score, 'img_msg': img_msg, 'txt': txt_score, 'txt_msg': txt_msg, 'proc': proc_score, 'proc_msg': proc_msg, 'total': (img_score + txt_score + proc_score) / 3}

    # -------------------------------------------------------------
    # 4. 구문 정확성 평가
    # -------------------------------------------------------------
    def _validate_syntax(self):
        # [이미지] cv2.imread()를 통해 이미지 바이너리 손상 여부 검증
        valid_img_count = sum(1 for p in self.df['FILEPATH'] if os.path.exists(str(p)) and cv2.imread(str(p)) is not None)
        vision_cols = [c for c in ['LOTID', 'FILEPATH', 'DATIME'] if c in self.df.columns]
        vision_missing_count = (~self.df[vision_cols].isnull().any(axis=1)).sum()
        
        img_score = (((valid_img_count / self.total_rows) * 100) + ((vision_missing_count / self.total_rows) * 100)) / 2
        img_msg = f"디코딩({valid_img_count}/{self.total_rows}) / 비전 컬럼({vision_missing_count}/{self.total_rows})"

        # [텍스트] ERRORTEXT 내 유니코드 깨짐 문자 및 비정상 문자열 감지
        cond_has_text = self.df['ERRORTEXT'].notnull() & (self.df['ERRORTEXT'].astype(str).str.strip() != "")
        df_text_target = self.df[cond_has_text]
        valid_text_count = (~df_text_target['ERRORTEXT'].astype(str).str.contains(r'[\ufffd]|[ㄱ-ㅎㅏ-ㅣ]', regex=True, na=False)).sum()
        txt_score = (valid_text_count / len(df_text_target) * 100) if len(df_text_target) > 0 else 100.0
        txt_msg = f"깨짐 문자 없음({valid_text_count}/{len(df_text_target)})"

        # [공정] Vision 데이터 필수 컬럼 및 공정 데이터 (PROC_TYPE별) 결측치 검증
        proc_score = 0.0
        proc_msg = "검증 대상 없음"
        if self.total_val_lots > 0:
            format_strings = ','.join(['%s'] * self.total_val_lots)
            try:
                df_proc = pd.read_sql(f"SELECT DAID, lotid, DATIME, PROC_TYPE, WELD_CURR, WELD_VOLT, WELD_TEMP, PAINT_PRESS, PAINT_TEMP, PAINT_HUMID FROM ai_proc_davalue WHERE lotid IN ({format_strings})", con=self.engine, params=tuple(self.validation_lots))
                cond_miss = df_proc[['DAID', 'lotid', 'DATIME', 'PROC_TYPE']].isnull().any(axis=1) | ((df_proc['PROC_TYPE'] == '용접') & df_proc[['WELD_CURR', 'WELD_VOLT', 'WELD_TEMP']].isnull().any(axis=1)) | ((df_proc['PROC_TYPE'] == '도장') & df_proc[['PAINT_PRESS', 'PAINT_TEMP', 'PAINT_HUMID']].isnull().any(axis=1))
                valid_proc_count = (~cond_miss).sum()
                proc_score = (valid_proc_count / len(df_proc) * 100) if len(df_proc) > 0 else 0.0
                proc_msg = f"센서 데이터 수집({valid_proc_count}/{len(df_proc)})"
            except: pass

        return {'img': img_score, 'img_msg': img_msg, 'txt': txt_score, 'txt_msg': txt_msg, 'proc': proc_score, 'proc_msg': proc_msg, 'total': (img_score + txt_score + proc_score) / 3}

    # -------------------------------------------------------------
    # 5. 통계적 다양성 평가
    # -------------------------------------------------------------
    def _validate_diversity(self):
        # [이미지] 이미지 밝기/대비 분포 (밝기 편차 > 5.0. 대비 편차 > 10.0 기준)
        b_list, c_list = [], []
        for p in self.df['FILEPATH']:
            if os.path.exists(str(p)):
                img = cv2.imread(str(p), cv2.IMREAD_GRAYSCALE)
                if img is not None:
                    m, s = cv2.meanStdDev(img)
                    b_list.append(m[0][0])
                    c_list.append(s[0][0])
                    
        is_b_diverse, is_c_diverse = False, False
        img_msg = "데이터 부족"
        if b_list and c_list:
            b_std, c_std = np.std(b_list), np.std(c_list)
            is_b_diverse, is_c_diverse = b_std > 5.0, c_std > 10.0
            img_score = 100.0 if (is_b_diverse and is_c_diverse) else 50.0
            pass_fail = "합격" if img_score == 100.0 else "불합격"
            img_msg = f"편차(밝기:{b_std:.1f}, 대비:{c_std:.1f}) -> {pass_fail} [기준: 밝기>5.0, 대비>10.0]"
        else:
            img_score = 50.0

        # [텍스트] ERRORTYPE 기준 그룹화하여 문장 길이 통계 산출 
        txt_score = 0.0
        txt_msg = "불량 데이터 부족"
        if 'ERRORTYPE' in self.df.columns and 'ERRORTEXT' in self.df.columns:
            df_err = self.df[self.df['ERRORTYPE'].notnull() & (self.df['ERRORTYPE'].astype(str).str.upper() != 'NONE')].copy()
            if not df_err.empty:
                df_err['LEN'] = df_err['ERRORTEXT'].astype(str).str.len()
                mean_len = df_err.groupby('ERRORTYPE')['LEN'].mean()
                
                if len(mean_len) > 0:
                    max_diff = mean_len.max() - mean_len.min()
                    txt_score = 100.0 if max_diff <= 10.0 else 50.0
                    pass_fail = "합격" if txt_score == 100.0 else "불합격"
                    txt_msg = f"유형별 최대 길이차:{max_diff:.1f}자 -> {pass_fail} [기준: 10자 이내]"
        # [공정] 정상/불량 비율 (불량 비율 20~70% 이내)
        def classify_status(val): 
            return 'NORMAL' if str(val).strip().upper() in ['정상'] else 'DEFECT'
        
        known = self.df[self.df['ISERROR'].notnull()]['ISERROR'].apply(classify_status)
        n_cnt, d_cnt = (known == 'NORMAL').sum(), (known == 'DEFECT').sum()
        
        null_lots = self.df[self.df['ISERROR'].isnull()]['LOTID'].dropna().unique().tolist()
        if null_lots:
            try:
                format_strings = ','.join(['%s'] * len(null_lots))
                query_pre = f"SELECT lotid, ISERROR FROM ai_proc_prevalue WHERE lotid IN ({format_strings})"
                df_pre = pd.read_sql(query_pre, con=self.engine, params=tuple(null_lots)).drop_duplicates(subset=['lotid'], keep='last')
                db_stat = df_pre['ISERROR'].apply(classify_status)
                n_cnt += (db_stat == 'NORMAL').sum()
                d_cnt += (db_stat == 'DEFECT').sum()
            except: 
                pass
            
        valid_counts = n_cnt + d_cnt
        proc_score = 50.0
        proc_msg = "분류 데이터 없음"
        
        if valid_counts > 0:
            defect_ratio = (d_cnt / valid_counts) * 100
            proc_score = 100.0 if 20.0 <= defect_ratio <= 70.0 else 50.0
            pass_fail = "합격" if proc_score == 100.0 else "불합격"
            proc_msg = f"정상 {n_cnt}건, 불량 {d_cnt}건(불량률:{defect_ratio:.1f}%) -> {pass_fail} [기준: 20~70%]"

        return {'img': img_score, 'img_msg': img_msg, 'txt': txt_score, 'txt_msg': txt_msg, 'proc': proc_score, 'proc_msg': proc_msg, 'total': (img_score + txt_score + proc_score) / 3}

    # -------------------------------------------------------------
    # 최종 결론 리포트 출력부
    # -------------------------------------------------------------
    def _print_final_report(self):
        avg_total = sum(s['total'] for s in self.scores.values()) / 5
        
        print("\n" + "="*70)
        print(f" [비정형 데이터 멀티모달 분류 모델 - 정합성 평가 최종 리포트]")
        print("="*70)
        print(f" 평가 대상: 검증 데이터셋 (총 {self.total_rows:,}건)\n")

        categories = [
            ('structure', '1. 데이터 구조 정합성', '파일명/물리파일, VLM 에러 매칭, 공정 DB 매핑 등 기본 구조 점검'),
            ('input_range', '2. 입력값 범위 정합성', '픽셀 배열, 사전 에러 키워드, 공정 센서 가동 범위 내 이상치 점검'),
            ('format', '3. 데이터 형식 정합성', '허용 확장자 준수, 템플릿 정규식 매칭, 공정 스키마 필수 컬럼 점검'),
            ('syntax', '4. 구문 정확성 평가', '이미지 디코딩, 유니코드 깨짐 문자 탐지, 필수 데이터 수집 누락 점검'),
            ('diversity', '5. 통계적 다양성 평가', '이미지 밝기/대비 편차, 텍스트 길이 다양성, 정상/불량 비율 균형 점검')
        ]

        for key, title, desc in categories:
            s = self.scores[key]
            print(f" [{title}] 종합: {s['total']:>6.2f}%")
            print(f"   - [이미지] {s['img']:>6.2f}% ({s['img_msg']})")
            print(f"   - [텍스트] {s['txt']:>6.2f}% ({s['txt_msg']})")
            print(f"   - [공정]   {s['proc']:>6.2f}% ({s['proc_msg']})")
            print(f"   * 목적: {desc}\n")

        print("-" * 70)
        print(f" >>> [최종 결론] 전체 파이프라인 종합 정합성 평균 : {avg_total:>6.2f}%")
        print("=" * 70 + "\n")


# -------------------------------------------------------------
# 파이프라인 연계 실행부
# -------------------------------------------------------------
if df_validation is not None:
    safe_password = urllib.parse.quote_plus(DB_PASSWORD)
    db_url = f"mysql+pymysql://{DB_USER}:{safe_password}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
    engine = create_engine(db_url)
    
    validator = MultimodalDataValidator(df_validation, engine)
    validator.run_all_validations()

-> [진행 중] 데이터 통합 정합성 평가 파이프라인 가동


KeyboardInterrupt: 